# 01 — Exploración de datos y área de estudio

**Paper:** AGC Chocó Andino (ECOLIND-60399)  
**Objetivo:** Leer el CSV de entrenamiento, analizar períodos temporales, distribución de labels, y generar el área de estudio como shapefile a partir del extent de los puntos y del FCD.

Este script **no descarga nada** de Earth Engine.

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from shapely.geometry import Point, box
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print('✓ Librerías cargadas')

✓ Librerías cargadas


## Rutas

In [13]:
# ── RUTAS ──────────────────────────────────────────────────────────────────
CSV_PATH        = Path(r'D:\PAPER-ECOLIN\DATA\raw\training_july_2025.csv')
FCD_PATH        = Path(r'D:\PAPER-ECOLIN\DATA\raw\FCD\landtrendr_fitted_prediction_2000.tif')
OUTPUT_DIR      = Path(r'D:\PAPER-ECOLIN\DATA\raw\AREA_STUDIO')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'CSV  : {CSV_PATH}')
print(f'FCD  : {FCD_PATH}')
print(f'Salida: {OUTPUT_DIR}')

CSV  : D:\PAPER-ECOLIN\DATA\raw\training_july_2025.csv
FCD  : D:\PAPER-ECOLIN\DATA\raw\FCD\landtrendr_fitted_prediction_2000.tif
Salida: D:\PAPER-ECOLIN\DATA\raw\AREA_STUDIO


---
## 1 — Leer el CSV

In [3]:
df = pd.read_csv(CSV_PATH)

print(f'Filas  : {len(df):,}')
print(f'Columnas: {df.shape[1]}')
print()
print('Columnas disponibles:')
print(df.columns.tolist())
print()
print('Primeras filas:')
df.head(3)

Filas  : 954
Columnas: 25

Columnas disponibles:
['unique_id_combined', 'code', 'unique_id_Avitabile', 'unique_id_GEDI', 'AGB', 'x', 'y', 'Year(s)_Avitabile', 'Year_ini_even', 'Year_end_even', 'Year_mean_Avitabile', 'Year_mean_even', 'elevation', 'fcd_average_period', 'ysc_rc_deg', 'ysc_rc_def', 'ysc_ft_deg', 'ysc_ft_def', 'Landsat_Blue', 'Landsat_Green', 'Landsat_Red', 'Landsat_NIR', 'Landsat_SWIR1', 'Landsat_SWIR2', 'HH_mean']

Primeras filas:


,unique_id_combined,code,unique_id_Avitabile,unique_id_GEDI,AGB,x,y,Year(s)_Avitabile,Year_ini_even,Year_end_even,...,ysc_rc_def,ysc_ft_deg,ysc_ft_def,Landsat_Blue,Landsat_Green,Landsat_Red,Landsat_NIR,Landsat_SWIR1,Landsat_SWIR2,HH_mean
0,386,CAM2,356,-386,81.78,-81.862821,9.119227,2008-2012,2008,2012,...,2,10,10,0.024750,0.044516,0.028475,0.356170,0.154966,0.064296,5297.8
1,360,CAM2,330,-360,89.58,-78.879487,8.410890,2008-2012,2008,2012,...,2,10,10,0.023620,0.053266,0.033417,0.397132,0.185171,0.074450,5760.5
2,454,CAM2,424,-454,74.87,-78.862823,8.310890,2008-2012,2008,2012,...,2,10,10,0.023346,0.050597,0.033929,0.372729,0.187530,0.078931,5474.5


In [4]:
# Resumen de nulos por columna clave
cols_key = ['x','y','AGB','code','Year_ini_even','Year_end_even',
            'Year_mean_Avitabile','Year_mean_even',
            'Landsat_Blue','Landsat_Green','Landsat_Red',
            'Landsat_NIR','Landsat_SWIR1','Landsat_SWIR2','HH_mean']

print('Nulos por columna clave:')
print(df[cols_key].isnull().sum().to_string())
print()
print('Rango de AGB (Mg/ha):')
print(df['AGB'].describe().round(2))

Nulos por columna clave:
x                       0
y                       0
AGB                     0
code                    0
Year_ini_even           0
Year_end_even           0
Year_mean_Avitabile     0
Year_mean_even          0
Landsat_Blue           37
Landsat_Green          37
Landsat_Red            37
Landsat_NIR            37
Landsat_SWIR1          37
Landsat_SWIR2          37
HH_mean                 0

Rango de AGB (Mg/ha):
count    954.00
mean     156.43
std       76.48
min       11.83
25%      115.72
50%      153.00
75%      208.92
max      497.45
Name: AGB, dtype: float64


---
## 2 — Períodos temporales (Year_ini_even → Year_end_even)

In [5]:
# Crear columna de período legible
df['periodo'] = (
    df['Year_ini_even'].astype(str) + '–' +
    df['Year_end_even'].astype(str)
)

# Contar puntos por período
periodos = (
    df.groupby(['Year_ini_even','Year_end_even','periodo'])
    .size()
    .reset_index(name='n_puntos')
    .sort_values('Year_ini_even')
)

# Marcar si es un punto único (ini == end) o un rango
periodos['tipo'] = periodos.apply(
    lambda r: 'año único' if r['Year_ini_even'] == r['Year_end_even'] else 'rango',
    axis=1
)

print(f'Períodos distintos: {len(periodos)}')
print()
print(periodos.to_string(index=False))
print()
print(f"Puntos con año único : {periodos.loc[periodos['tipo']=='año único','n_puntos'].sum():,}")
print(f"Puntos con rango     : {periodos.loc[periodos['tipo']=='rango','n_puntos'].sum():,}")

Períodos distintos: 29

 Year_ini_even  Year_end_even   periodo  n_puntos      tipo
          2000           2000 2000–2000         9 año único
          2002           2002 2002–2002         9 año único
          2002           2004 2002–2004         1     rango
          2002           2006 2002–2006         9     rango
          2002           2012 2002–2012         2     rango
          2004           2004 2004–2004         9 año único
          2004           2006 2004–2006         1     rango
          2004           2008 2004–2008         2     rango
          2004           2012 2004–2012         3     rango
          2004           2014 2004–2014         3     rango
          2006           2006 2006–2006         9 año único
          2006           2008 2006–2008         2     rango
          2006           2010 2006–2010         1     rango
          2006           2012 2006–2012         4     rango
          2006           2014 2006–2014         3     rango
          2006  

In [6]:
# Distribución de Year_mean_even (el año que usaremos para asignar Landsat)
print('Distribución de Year_mean_even (año central del período):')
print(df['Year_mean_even'].value_counts().sort_index().to_string())
print()

# Rango total de datos
print(f"Año más antiguo (ini): {df['Year_ini_even'].min()}")
print(f"Año más reciente (end): {df['Year_end_even'].max()}")

Distribución de Year_mean_even (año central del período):
Year_mean_even
2000      9
2002      9
2004     19
2006     15
2008     14
2010    477
2012     27
2014     32
2016     40
2018     24
2020    282
2022      6

Año más antiguo (ini): 2000
Año más reciente (end): 2022


---
## 3 — Distribución por label (`code`)

In [7]:
# Conteo por código/label
labels = (
    df.groupby('code')
    .agg(
        n_puntos=('code','count'),
        agb_media=('AGB','mean'),
        agb_min=('AGB','min'),
        agb_max=('AGB','max'),
        agb_std=('AGB','std')
    )
    .round(1)
    .reset_index()
    .sort_values('n_puntos', ascending=False)
)

labels['pct'] = (labels['n_puntos'] / len(df) * 100).round(1)

print(f'Códigos/labels distintos: {len(labels)}')
print()
print(labels.to_string(index=False))

Códigos/labels distintos: 8

code  n_puntos  agb_media  agb_min  agb_max  agb_std  pct
CAM2       290      138.5     15.0    288.1     75.2 30.4
GEDI       284      141.1     15.0    241.0     47.5 29.8
SAM3       107      231.9     15.0    497.4     79.5 11.2
 RBA        68      143.6     15.0    410.0     87.8  7.1
SAM5        56      197.0     50.9    275.7     58.4  5.9
SAM4        56      182.4     15.0    272.5     88.1  5.9
USFQ        53      188.2     98.0    413.4     66.0  5.6
  BP        40       80.4     11.8    160.4     45.5  4.2


In [8]:
# Identificar los weak-labels (AGB == 15 que mencionan los revisores)
weak = df[df['AGB'] == 15.0]
print(f'Filas con AGB exactamente = 15 (weak-labels): {len(weak)}')
if len(weak) > 0:
    print('  Distribución por code:')
    print(weak['code'].value_counts().to_string())
    print()
    # Cuántos están dentro de la reserva (coordenadas del Chocó Andino)
    # Bbox aproximado del Chocó Andino: lon -78.9 a -78.4, lat -0.3 a 0.1
    chocho_bbox = dict(xmin=-78.9, xmax=-78.4, ymin=-0.3, ymax=0.1)
    weak_chocho = weak[
        (weak['x'] >= chocho_bbox['xmin']) & (weak['x'] <= chocho_bbox['xmax']) &
        (weak['y'] >= chocho_bbox['ymin']) & (weak['y'] <= chocho_bbox['ymax'])
    ]
    print(f'  Weak-labels dentro del bbox Chocó Andino: {len(weak_chocho)}')
    print(f'  Weak-labels fuera del bbox Chocó Andino : {len(weak) - len(weak_chocho)}')

Filas con AGB exactamente = 15 (weak-labels): 86
  Distribución por code:
code
CAM2    42
GEDI    21
RBA     10
SAM4     9
SAM3     4

  Weak-labels dentro del bbox Chocó Andino: 5
  Weak-labels fuera del bbox Chocó Andino : 81


---
## 4 — Área de estudio: extent de los puntos → shapefile WGS84

In [9]:
# ── 4a. GeoDataFrame de todos los puntos ───────────────────────────────────
gdf_pts = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['x'], df['y']),
    crs='EPSG:4326'
)

# Extent total de todos los puntos
xmin_all, ymin_all, xmax_all, ymax_all = gdf_pts.total_bounds
print('Extent de TODOS los puntos del CSV (WGS84):')
print(f'  x (lon): {xmin_all:.6f} → {xmax_all:.6f}')
print(f'  y (lat): {ymin_all:.6f} → {ymax_all:.6f}')
print()

Extent de TODOS los puntos del CSV (WGS84):
  x (lon): -82.921153 → -60.529488
  y (lat): -17.089108 → 10.527559



In [10]:
# ── 4b. Bbox del Chocó Andino (solo puntos dentro de la reserva) ───────────
# Filtramos los puntos que están dentro del bbox aproximado de la reserva.
# Ajusta estos límites si tienes coordenadas exactas.
CHOCHO_XMIN = -78.95
CHOCHO_XMAX = -78.30
CHOCHO_YMIN = -0.35
CHOCHO_YMAX =  0.15

gdf_chocho = gdf_pts[
    (gdf_pts['x'] >= CHOCHO_XMIN) & (gdf_pts['x'] <= CHOCHO_XMAX) &
    (gdf_pts['y'] >= CHOCHO_YMIN) & (gdf_pts['y'] <= CHOCHO_YMAX)
].copy()

print(f'Puntos dentro del bbox Chocó Andino: {len(gdf_chocho):,} de {len(df):,}')

if len(gdf_chocho) > 0:
    xmin_c, ymin_c, xmax_c, ymax_c = gdf_chocho.total_bounds
    print(f'  Extent Chocó Andino:')
    print(f'    lon: {xmin_c:.6f} → {xmax_c:.6f}')
    print(f'    lat: {ymin_c:.6f} → {ymax_c:.6f}')

Puntos dentro del bbox Chocó Andino: 264 de 954
  Extent Chocó Andino:
    lon: -78.949000 → -78.416973
    lat: -0.348434 → 0.148983


In [11]:
# ── 4c. Guardar shapefile: extent de TODOS los puntos ──────────────────────
bbox_all = box(xmin_all, ymin_all, xmax_all, ymax_all)
gdf_bbox_all = gpd.GeoDataFrame(
    [{'source': 'CSV_all_points',
      'n_pts': len(df),
      'xmin': xmin_all, 'ymin': ymin_all,
      'xmax': xmax_all, 'ymax': ymax_all}],
    geometry=[bbox_all],
    crs='EPSG:4326'
)

out_all = OUTPUT_DIR / 'study_area_all_points.shp'
gdf_bbox_all.to_file(out_all)
print(f'✓ Shapefile guardado: {out_all}')

# ── 4d. Guardar shapefile: extent solo puntos Chocó Andino ─────────────────
if len(gdf_chocho) > 0:
    bbox_c = box(xmin_c, ymin_c, xmax_c, ymax_c)
    gdf_bbox_chocho = gpd.GeoDataFrame(
        [{'source': 'CSV_chocho_andino',
          'n_pts': len(gdf_chocho),
          'xmin': xmin_c, 'ymin': ymin_c,
          'xmax': xmax_c, 'ymax': ymax_c}],
        geometry=[bbox_c],
        crs='EPSG:4326'
    )
    out_chocho = OUTPUT_DIR / 'study_area_chocho_andino.shp'
    gdf_bbox_chocho.to_file(out_chocho)
    print(f'✓ Shapefile guardado: {out_chocho}')

# ── 4e. Guardar también los puntos completos como shapefile ────────────────
out_pts = OUTPUT_DIR / 'training_points_all.shp'
gdf_pts[['unique_id_combined','code','AGB','x','y',
          'Year_ini_even','Year_end_even','Year_mean_even','geometry']].to_file(out_pts)
print(f'✓ Puntos guardados: {out_pts}')

✓ Shapefile guardado: D:\PAPER-ECOLIN\DATA\raw\AREA_STUDIO\study_area_all_points.shp
✓ Shapefile guardado: D:\PAPER-ECOLIN\DATA\raw\AREA_STUDIO\study_area_chocho_andino.shp
✓ Puntos guardados: D:\PAPER-ECOLIN\DATA\raw\AREA_STUDIO\training_points_all.shp


---
## 5 — Extent del FCD → shapefile `study_area`

In [14]:
# Leer el FCD y extraer su extent
with rasterio.open(FCD_PATH) as src:
    fcd_crs   = src.crs
    fcd_bounds = src.bounds          # left, bottom, right, top
    fcd_transform = src.transform
    fcd_shape  = src.shape           # (rows, cols)
    fcd_res    = (abs(src.transform.a), abs(src.transform.e))  # (xres, yres)
    fcd_nodata = src.nodata
    fcd_count  = src.count
    fcd_dtype  = src.dtypes

print('FCD raster info:')
print(f'  CRS      : {fcd_crs}')
print(f'  Resolución: {fcd_res[0]:.2f} x {fcd_res[1]:.2f} m (aprox.)')
print(f'  Filas x Cols: {fcd_shape[0]:,} x {fcd_shape[1]:,}')
print(f'  Bandas   : {fcd_count}')
print(f'  Dtype    : {fcd_dtype}')
print(f'  NoData   : {fcd_nodata}')
print(f'  Bounds (nativo): {fcd_bounds}')

FCD raster info:
  CRS      : EPSG:4326
  Resolución: 0.00 x 0.00 m (aprox.)
  Filas x Cols: 2,817 x 2,816
  Bandas   : 1
  Dtype    : ('float64',)
  NoData   : None
  Bounds (nativo): BoundingBox(left=-79.07186726488217, bottom=-0.3969655240524166, right=-78.312970512858, top=0.3622007225569911)


In [15]:
from rasterio.warp import transform_bounds

# Reproyectar extent del FCD a WGS84 si no está en 4326
if fcd_crs and fcd_crs.to_epsg() != 4326:
    bounds_wgs84 = transform_bounds(
        fcd_crs, 'EPSG:4326',
        fcd_bounds.left, fcd_bounds.bottom,
        fcd_bounds.right, fcd_bounds.top
    )
    print(f'Bounds reproyectados a WGS84: {bounds_wgs84}')
else:
    bounds_wgs84 = (fcd_bounds.left, fcd_bounds.bottom,
                    fcd_bounds.right, fcd_bounds.top)
    print(f'FCD ya está en WGS84: {bounds_wgs84}')

fcd_xmin, fcd_ymin, fcd_xmax, fcd_ymax = bounds_wgs84
print()
print(f'Extent FCD en WGS84:')
print(f'  lon: {fcd_xmin:.6f} → {fcd_xmax:.6f}')
print(f'  lat: {fcd_ymin:.6f} → {fcd_ymax:.6f}')

FCD ya está en WGS84: (-79.07186726488217, -0.3969655240524166, -78.312970512858, 0.3622007225569911)

Extent FCD en WGS84:
  lon: -79.071867 → -78.312971
  lat: -0.396966 → 0.362201


In [16]:
# Guardar el extent del FCD como shapefile study_area
fcd_box = box(fcd_xmin, fcd_ymin, fcd_xmax, fcd_ymax)

gdf_study_area = gpd.GeoDataFrame(
    [{
        'source'   : 'FCD_2000_extent',
        'raster'   : FCD_PATH.name,
        'crs_orig' : str(fcd_crs),
        'xmin'     : fcd_xmin,
        'ymin'     : fcd_ymin,
        'xmax'     : fcd_xmax,
        'ymax'     : fcd_ymax,
        'rows'     : fcd_shape[0],
        'cols'     : fcd_shape[1],
    }],
    geometry=[fcd_box],
    crs='EPSG:4326'
)

out_study = OUTPUT_DIR / 'study_area.shp'
gdf_study_area.to_file(out_study)
print(f'✓ study_area guardado: {out_study}')

✓ study_area guardado: D:\PAPER-ECOLIN\DATA\raw\AREA_STUDIO\study_area.shp


---
## 6 — Resumen final

In [17]:
print('=' * 60)
print('RESUMEN')
print('=' * 60)
print(f'Total registros CSV          : {len(df):,}')
print(f'Períodos distintos           : {len(periodos)}')
print(f'  - Año único                : {(periodos["tipo"]=="año único").sum()}')
print(f'  - Rango multi-año          : {(periodos["tipo"]=="rango").sum()}')
print(f'Labels (code) distintos      : {df["code"].nunique()}')
print(f'Weak-labels (AGB=15)         : {len(weak)}')
print()
print('Archivos generados:')
for f in sorted(OUTPUT_DIR.glob('*')):
    print(f'  {f.name}')
print('=' * 60)
print('✓ Script completado — no se descargó nada de GEE')

RESUMEN
Total registros CSV          : 954
Períodos distintos           : 29
  - Año único                : 12
  - Rango multi-año          : 17
Labels (code) distintos      : 8
Weak-labels (AGB=15)         : 86

Archivos generados:
  study_area.cpg
  study_area.dbf
  study_area.prj
  study_area.shp
  study_area.shx
  study_area_all_points.cpg
  study_area_all_points.dbf
  study_area_all_points.prj
  study_area_all_points.shp
  study_area_all_points.shx
  study_area_chocho_andino.cpg
  study_area_chocho_andino.dbf
  study_area_chocho_andino.prj
  study_area_chocho_andino.shp
  study_area_chocho_andino.shx
  training_points_all.cpg
  training_points_all.dbf
  training_points_all.prj
  training_points_all.shp
  training_points_all.shx
✓ Script completado — no se descargó nada de GEE
